# Contrastive Probe Inference

Runs the spatial-grounding probe: for each cached scene whose instruction
contains a swappable spatial term, the original and term-swapped instructions
are both passed to OpenVLA against the same frame, and both predictions are
logged with `pair_id`, `role`, and `scene_id`, the schema required by the
export notebook.

Predictions are written to a fresh CSV (`probe_predictions.csv`).

**Colab GPU:** prefer L4 or A100. Requires notebook 01's install and restart to
have been done in this session, and notebook 02's cache to exist on Drive.


## 1. Mount Drive


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
CACHE_DIR = '/content/drive/MyDrive/openvla_cache/bridge_multiobj'
PROBE_CSV = '/content/drive/MyDrive/openvla_cache/probe_predictions.csv'
print('cache ->', CACHE_DIR)
print('log   ->', PROBE_CSV)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache -> /content/drive/MyDrive/openvla_cache/bridge_multiobj
log   -> /content/drive/MyDrive/openvla_cache/probe_predictions.csv


## 2. Get the code and import


In [2]:
import sys, shutil, os, glob, importlib
REPO_DIR = '/content/ECS8056'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -q https://github.com/LewisTL/ECS8056.git {REPO_DIR}

module_dir = os.path.dirname(glob.glob(os.path.join(REPO_DIR, '**', 'model.py'), recursive=True)[0])
if module_dir not in sys.path:
    sys.path.insert(0, module_dir)
for m in ('model', 'data'):
    sys.modules.pop(m, None)
importlib.invalidate_caches()

from model import load_openvla, predict_action, run_metadata, append_prediction_log
from data import load_manifest
print('imported model.py and data.py from', module_dir)

imported model.py and data.py from /content/ECS8056


## 3. Load OpenVLA-7B


In [3]:
processor, vla, compute_dtype = load_openvla(quantize_4bit=True, precision='bf16')
meta = run_metadata(compute_dtype)
print(meta)

[load_openvla] GPU: NVIDIA A100-SXM4-80GB (sm_80, 79.3 GB) | precision=bf16 | attn=eager | 4bit=True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:99: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[load_openvla] Loaded. GPU memory allocated: 4.08 GB
{'gpu_name': 'NVIDIA A100-SXM4-80GB', 'gpu_capability': 'sm_80', 'dtype': 'bfloat16', 'seed': 42, 'torch': '2.11.0+cu128', 'transformers': '4.40.1', 'bitsandbytes': '0.49.2'}


## 4. Generate contrastive pairs from the manifest

Minimal-pair construction by antonym swap: a scene qualifies when its
instruction contains exactly one swappable spatial term, so the pair differs
in that term alone. Instructions with zero or multiple swappable terms are
excluded (a multi-term swap would change more than one relation and break the
minimal-pair property). Role `a` is always the original instruction; role `b`
is the swapped variant.


In [4]:
import re

SWAP_MAP = {
    'left': 'right', 'right': 'left',
    'leftmost': 'rightmost', 'rightmost': 'leftmost',
    'front': 'back', 'back': 'front',
    'top': 'bottom', 'bottom': 'top',
    'nearest': 'farthest', 'farthest': 'nearest',
    'closer': 'farther', 'farther': 'closer',
}
_SWAP_RE = re.compile(r'\b(' + '|'.join(SWAP_MAP) + r')\b')

def make_pair(instruction: str):
    """Return (term, swapped_instruction) if exactly one swappable term occurs."""
    hits = _SWAP_RE.findall(instruction.lower())
    if len(hits) != 1:
        return None
    term = hits[0]
    swapped = re.sub(r'\b' + term + r'\b', SWAP_MAP[term], instruction,
                     flags=re.IGNORECASE)
    return term, swapped

rows = load_manifest(CACHE_DIR)
probe_set = []
for row in rows:
    made = make_pair(row['instruction'])
    if made is None:
        continue
    term, swapped = made
    probe_set.append({
        'scene_id': row['episode_index'],
        'pair_id': f"ep{int(row['episode_index']):06d}_{term}",
        'spatial_term': term,
        'image_path': os.path.join(CACHE_DIR, row['image_path']),
        'instr_a': row['instruction'],
        'instr_b': swapped,
    })

print(f'{len(rows)} cached scenes -> {len(probe_set)} probe pairs')
for p in probe_set[:5]:
    print(f"[{p['pair_id']}]")
    print('  a:', p['instr_a'])
    print('  b:', p['instr_b'])

224 cached scenes -> 68 probe pairs
[ep000000_left]
  a: Place the can to the left of the pot.
  b: Place the can to the right of the pot.
[ep000002_front]
  a: Slide the cloth diagonally to the front of the spoon
  b: Slide the cloth diagonally to the back of the spoon
[ep000004_right]
  a: Move the kadai and place it at the right edge of the table.
  b: Move the kadai and place it at the left edge of the table.
[ep000014_top]
  a: Move the Orange cloth towards the top of the table
  b: Move the Orange cloth towards the bottom of the table
[ep000017_front]
  a: Move the colander in front of the red spoon
  b: Move the colander in back of the red spoon


## 5. Run the probe

Two deterministic predictions per pair (same frame, both instructions), each
logged with the pairing columns. `sample_idx` is fixed at 0 under the
deterministic decoding strategy; the column exists so the schema does not
change if repeated sampling or paraphrase variants are added later.

Restart-safe: pairs already present in the log are skipped, so an interrupted
run resumes where it stopped.


In [5]:
import csv
import numpy as np
from PIL import Image

done = set()
if os.path.exists(PROBE_CSV):
    with open(PROBE_CSV, newline='') as f:
        done = {r['pair_id'] for r in csv.DictReader(f)}
    print(f'resuming: {len(done)} pair ids already logged')

for i, p in enumerate(probe_set):
    if p['pair_id'] in done:
        continue
    image = Image.open(p['image_path'])
    for role, instr in (('a', p['instr_a']), ('b', p['instr_b'])):
        action = predict_action(processor, vla, image, instr, compute_dtype)
        append_prediction_log(
            PROBE_CSV, action, instr, meta,
            scene_id=p['scene_id'],
            pair_id=p['pair_id'],
            role=role,
            spatial_term=p['spatial_term'],
            sample_idx=0,
        )
    if (i + 1) % 10 == 0:
        print(f'{i + 1}/{len(probe_set)} pairs done')

print('probe complete ->', PROBE_CSV)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


10/68 pairs done
20/68 pairs done
30/68 pairs done
40/68 pairs done
50/68 pairs done
60/68 pairs done
probe complete -> /content/drive/MyDrive/openvla_cache/probe_predictions.csv


## 6. Quick directional read

A sanity read of the x-axis sign-flip rate for left/right
pairs before the export/pilot notebook runs. The formal metrics, ground-truth
validation, and frame checks belong to the next notebook.


In [6]:
import pandas as pd
log = pd.read_csv(PROBE_CSV)
lr = log[log['spatial_term'].isin(['left', 'right'])]
wide = lr.pivot_table(index='pair_id', columns='role', values='a0')
flips = (wide['a'] * wide['b'] < 0)
print(f"left/right pairs: {len(wide)} | dx sign flips: {flips.sum()} "
      f"({flips.mean():.1%})")
wide.head(8)

left/right pairs: 49 | dx sign flips: 8 (16.3%)


role,a,b
pair_id,,
ep000000_left,-0.002669,-0.002669
ep000004_right,-0.014077,-0.012958
ep000027_left,-0.002669,-0.002669
ep000031_left,-0.000209,-0.000209
ep000033_left,-0.004011,-0.004011
ep000036_right,-0.003788,0.000462
ep000037_right,-0.001327,-0.001327
ep000043_left,-0.002446,-0.002446
